# gnn_structure_removal_experiment
GCN-focused structure-removal study with step-by-step plot generation.

## Setup


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from notebook_runner import run_notebook_script


## Configuration


In [ ]:
SCRIPT_NAME = '04_gcn_structure_removal'
overrides = {
    'render_all_plots': False,  # plot figure-by-figure below
}


## Compute Shared State


In [ ]:
state = run_notebook_script(SCRIPT_NAME, overrides=overrides)
print('Plot dir:', state['PLOT_DIR'])
print('Final test acc:', float(state['final_test_acc']))

print('Palette:', {'eventblue': state['EVENT_BLUE'], 'snapshotorange': state['SNAPSHOT_ORANGE'], 'edgegray': state['EDGE_GRAY']})

### Plot 1: SBM Graph (Node Colors = Class)


In [ ]:
state['draw_graph'](state['G'], state['y'], title='SBM graph (node color = class)', save_as='sbm_graph')


### Plot 2: Training Loss Curve


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(state['history']['epoch'], state['history']['loss'], color=state['EVENT_BLUE'])
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('Loss curve')
state['savefig_pdf']('loss_curve')
plt.show()


### Plot 3: Accuracy Curves (Train/Val/Test)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(state['history']['epoch'], state['history']['train_acc'], label='train', color=state['EVENT_BLUE'])
plt.plot(state['history']['epoch'], state['history']['val_acc'], label='val', color=state['SNAPSHOT_ORANGE'])
plt.plot(state['history']['epoch'], state['history']['test_acc'], label='test', color=state['EDGE_GRAY'])
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy curves')
state['savefig_pdf']('accuracy_curves')
plt.legend()
plt.show()


### Plot 4: PCA of Hidden Embeddings (Full vs Self-Loop-Only)


In [ ]:
Zf, Zs = state['_project_pair'](state['h_full'], state['h_self'], method='pca')
state['_plot_full_vs_selfloop'](
    Zf,
    Zs,
    state['y_d'],
    title='PCA: hidden embeddings (full vs self-loop)',
    save_name='pca_full_vs_selfloop',
)


### Plot 5: PCA of Logits (Full vs Self-Loop-Only)


In [ ]:
Zf, Zs = state['_project_pair'](state['logits_full'], state['logits_self'], method='pca')
state['_plot_full_vs_selfloop'](
    Zf,
    Zs,
    state['y_d'],
    title='PCA: logits (full vs self-loop)',
    save_name='pca_logits_full_vs_selfloop',
)


### Plot 6: t-SNE of Hidden Embeddings (Full vs Self-Loop-Only)


In [ ]:
Zf, Zs = state['_project_pair'](state['h_full'], state['h_self'], method='tsne')
state['_plot_full_vs_selfloop'](
    Zf,
    Zs,
    state['y_d'],
    title='t-SNE: hidden embeddings (full vs self-loop)',
    save_name='tsne_full_vs_selfloop',
)


### Plot 7: Variant Summary Table


In [ ]:
df = state['df'].copy()
try:
    from IPython.display import display
    display(df)
except Exception:
    print(df.to_string())


### Plot 8: Accuracy vs Graph Variant


In [ ]:
df = state['df']
plt.figure(figsize=(7, 4))
plt.bar(df.index, df['acc'], color=state['EVENT_BLUE'])
plt.ylabel('Test accuracy')
plt.title('Accuracy vs graph variant')
state['savefig_pdf']('accuracy_vs_graph_variant')
plt.xticks(rotation=20, ha='right')
plt.show()


### Plot 9: Prediction Instability vs Original


In [ ]:
df = state['df']
plt.figure(figsize=(7, 4))
plt.bar(df.index, df['changed_frac'], color=state['SNAPSHOT_ORANGE'])
plt.ylabel('Fraction of nodes changed')
plt.title('Prediction instability vs original')
state['savefig_pdf']('changed_frac_vs_original')
plt.xticks(rotation=20, ha='right')
plt.show()


### Plot 10: Accuracy vs Removing Intra-Class Edges


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(state['fractions'], state['accs'], marker='o', color=state['EVENT_BLUE'])
plt.xlabel('Fraction of intra-class edges removed')
plt.ylabel('Test accuracy')
plt.title('Accuracy vs removing community structure')
state['savefig_pdf']('accuracy_vs_remove_intra')
plt.ylim(0, 1.0)
plt.grid(True, color=state['EDGE_GRAY'], alpha=0.3)
plt.show()


### Plot 11: Changed Predictions vs Removing Intra-Class Edges


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(state['fractions'], state['changeds'], marker='o', color=state['SNAPSHOT_ORANGE'])
plt.xlabel('Fraction of intra-class edges removed')
plt.ylabel('Fraction of nodes changed vs original')
plt.title('Prediction changes vs removing community structure')
state['savefig_pdf']('changed_vs_remove_intra')
plt.ylim(0, 1.0)
plt.grid(True, color=state['EDGE_GRAY'], alpha=0.3)
plt.show()
